# Fine-Tune T5 with LoRA for Business Rule to JSON Schema Conversion

This notebook fine-tunes a T5 model using LoRA (Low-Rank Adaptation) to convert
natural language business rules into JSON Schema constraints.

**Examples:**
- "Must be 18 or older" → `{"minimum": 18}`
- "Maximum 255 characters" → `{"maxLength": 255}`
- "Valid values: A, B, C" → `{"enum": ["A", "B", "C"]}`

## Setup
1. Runtime → Change runtime type → T4 GPU
2. Run all cells in order
3. Download the trained adapter at the end

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes
!pip install -q sentencepiece  # Required for T5 tokenizer

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# Check GPU availability
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

Choose your domain and model size.

In [ ]:
# ============================================================
# CONFIGURATION - Modify these settings as needed
# ============================================================

# Domain to train (change this to train different adapters)
DOMAIN = "financial"  # Options: "financial", "healthcare"

# Model size (t5-small is fastest, t5-base is more accurate)
MODEL_NAME = "t5-small"  # Options: "t5-small" (60M), "t5-base" (220M)

# Training parameters
EPOCHS = 10
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
MAX_INPUT_LENGTH = 128
MAX_OUTPUT_LENGTH = 128

# LoRA parameters
LORA_R = 16          # Rank of the update matrices
LORA_ALPHA = 32      # Scaling factor
LORA_DROPOUT = 0.1   # Dropout probability

# Output directory
OUTPUT_DIR = f"./adapter_{DOMAIN}"

print(f"Training {DOMAIN} adapter using {MODEL_NAME}")

## 3. Load Training Data

You can either upload your JSON files or paste the data directly.

In [ ]:
# Option 1: Upload JSON file from your computer
# Uncomment and run this cell to upload

# from google.colab import files
# uploaded = files.upload()
# filename = list(uploaded.keys())[0]
# with open(filename, 'r') as f:
#     data = json.load(f)
# examples = data['examples']

In [ ]:
# Option 2: Load from GitHub (replace with your repo URL)
# Uncomment and modify the URL

# import requests
# url = "https://raw.githubusercontent.com/YOUR_USER/constraints_2_json/main/training_data/financial_domain.json"
# data = requests.get(url).json()
# examples = data['examples']

In [ ]:
# Option 3: Embedded sample data (for testing)
# This contains a subset of examples - replace with full dataset for production

SAMPLE_DATA = {
    "financial": [
        {"input": "Balance must not be negative", "output": {"minimum": 0}},
        {"input": "Balance cannot be negative", "output": {"minimum": 0}},
        {"input": "Balance must be zero or greater", "output": {"minimum": 0}},
        {"input": "Account number must be exactly 10 digits", "output": {"pattern": "^[0-9]{10}$"}},
        {"input": "Account number is 10 numeric characters", "output": {"pattern": "^[0-9]{10}$"}},
        {"input": "Routing number must be 9 digits", "output": {"pattern": "^[0-9]{9}$"}},
        {"input": "Interest rate between 0 and 30 percent", "output": {"minimum": 0, "maximum": 30}},
        {"input": "Interest rate must be from 0 to 30", "output": {"minimum": 0, "maximum": 30}},
        {"input": "Rate cannot exceed 30 percent", "output": {"maximum": 30}},
        {"input": "Transaction type: DEBIT, CREDIT, TRANSFER", "output": {"enum": ["DEBIT", "CREDIT", "TRANSFER"]}},
        {"input": "Transaction type must be DEBIT, CREDIT, or TRANSFER", "output": {"enum": ["DEBIT", "CREDIT", "TRANSFER"]}},
        {"input": "Account type: CHECKING, SAVINGS, MONEY_MARKET, CD", "output": {"enum": ["CHECKING", "SAVINGS", "MONEY_MARKET", "CD"]}},
        {"input": "Minimum deposit is $100", "output": {"minimum": 100}},
        {"input": "Deposit must be at least $100", "output": {"minimum": 100}},
        {"input": "Maximum withdrawal is $10000 per day", "output": {"maximum": 10000}},
        {"input": "Withdrawal cannot exceed $10,000", "output": {"maximum": 10000}},
        {"input": "Credit score must be between 300 and 850", "output": {"minimum": 300, "maximum": 850}},
        {"input": "Loan term must be 12, 24, 36, 48, or 60 months", "output": {"enum": [12, 24, 36, 48, 60]}},
        {"input": "Currency code must be 3 uppercase letters", "output": {"pattern": "^[A-Z]{3}$"}},
        {"input": "Currency: USD, EUR, GBP, CAD, JPY", "output": {"enum": ["USD", "EUR", "GBP", "CAD", "JPY"]}},
        {"input": "SSN format: XXX-XX-XXXX (9 digits with dashes)", "output": {"pattern": "^[0-9]{3}-[0-9]{2}-[0-9]{4}$"}},
        {"input": "Credit card number must be 16 digits", "output": {"pattern": "^[0-9]{16}$"}},
        {"input": "CVV must be 3 or 4 digits", "output": {"pattern": "^[0-9]{3,4}$"}},
        {"input": "Age must be 18 or older", "output": {"minimum": 18}},
        {"input": "Customer must be at least 18 years old", "output": {"minimum": 18}},
        {"input": "ZIP code must be 5 digits", "output": {"pattern": "^[0-9]{5}$"}},
        {"input": "Phone number must be 10 digits", "output": {"pattern": "^[0-9]{10}$"}},
        {"input": "State code must be 2 uppercase letters", "output": {"pattern": "^[A-Z]{2}$"}},
        {"input": "Email must be valid format", "output": {"format": "email"}},
        {"input": "Date must be in ISO format", "output": {"format": "date"}},
        {"input": "Account holder name maximum 100 characters", "output": {"maxLength": 100}},
        {"input": "Name cannot exceed 100 characters", "output": {"maxLength": 100}},
        {"input": "Description must be at least 10 characters", "output": {"minLength": 10}},
        {"input": "Percentage must be between 0 and 100", "output": {"minimum": 0, "maximum": 100}},
        {"input": "Price must be greater than 0", "output": {"exclusiveMinimum": 0}},
        {"input": "Quantity must be a positive integer", "output": {"minimum": 1}},
    ],
    "healthcare": [
        {"input": "Patient age must be between 0 and 150", "output": {"minimum": 0, "maximum": 150}},
        {"input": "Age cannot be negative", "output": {"minimum": 0}},
        {"input": "MRN must be 8 alphanumeric characters", "output": {"pattern": "^[A-Z0-9]{8}$"}},
        {"input": "Medical Record Number: 8 characters", "output": {"pattern": "^[A-Z0-9]{8}$"}},
        {"input": "Blood type: A+, A-, B+, B-, O+, O-, AB+, AB-", "output": {"enum": ["A+", "A-", "B+", "B-", "O+", "O-", "AB+", "AB-"]}},
        {"input": "Dosage cannot exceed 1000mg", "output": {"maximum": 1000}},
        {"input": "Maximum dosage is 1000 mg", "output": {"maximum": 1000}},
        {"input": "Dosage must be between 0.1 and 500 mg", "output": {"minimum": 0.1, "maximum": 500}},
        {"input": "Dosage must be positive", "output": {"exclusiveMinimum": 0}},
        {"input": "Gender: Male, Female, Other, Unknown", "output": {"enum": ["Male", "Female", "Other", "Unknown"]}},
        {"input": "Patient status: ACTIVE, INACTIVE, DECEASED, MERGED", "output": {"enum": ["ACTIVE", "INACTIVE", "DECEASED", "MERGED"]}},
        {"input": "NPI must be exactly 10 digits", "output": {"pattern": "^[0-9]{10}$"}},
        {"input": "National Provider Identifier: 10 digits", "output": {"pattern": "^[0-9]{10}$"}},
        {"input": "CPT code must be 5 digits", "output": {"pattern": "^[0-9]{5}$"}},
        {"input": "Heart rate must be between 30 and 250 bpm", "output": {"minimum": 30, "maximum": 250}},
        {"input": "Systolic BP must be between 50 and 300", "output": {"minimum": 50, "maximum": 300}},
        {"input": "Temperature must be between 90 and 110 degrees F", "output": {"minimum": 90, "maximum": 110}},
        {"input": "Oxygen saturation must be between 0 and 100", "output": {"minimum": 0, "maximum": 100}},
        {"input": "Pain scale must be between 0 and 10", "output": {"minimum": 0, "maximum": 10}},
        {"input": "Glasgow Coma Scale: 3-15", "output": {"minimum": 3, "maximum": 15}},
        {"input": "Weight must be positive", "output": {"exclusiveMinimum": 0}},
        {"input": "BMI must be between 10 and 100", "output": {"minimum": 10, "maximum": 100}},
        {"input": "Frequency: ONCE, DAILY, BID, TID, QID, PRN", "output": {"enum": ["ONCE", "DAILY", "BID", "TID", "QID", "PRN"]}},
        {"input": "Route: PO, IV, IM, SC, SL, PR, TOP, INH", "output": {"enum": ["PO", "IV", "IM", "SC", "SL", "PR", "TOP", "INH"]}},
        {"input": "Allergy severity: MILD, MODERATE, SEVERE, LIFE_THREATENING", "output": {"enum": ["MILD", "MODERATE", "SEVERE", "LIFE_THREATENING"]}},
        {"input": "Refills must be between 0 and 12", "output": {"minimum": 0, "maximum": 12}},
        {"input": "Days supply must be between 1 and 365", "output": {"minimum": 1, "maximum": 365}},
        {"input": "Patient name maximum 200 characters", "output": {"maxLength": 200}},
        {"input": "Name cannot be empty", "output": {"minLength": 1}},
        {"input": "Notes maximum 5000 characters", "output": {"maxLength": 5000}},
        {"input": "SSN format: XXX-XX-XXXX", "output": {"pattern": "^[0-9]{3}-[0-9]{2}-[0-9]{4}$"}},
        {"input": "Phone number must be 10 digits", "output": {"pattern": "^[0-9]{10}$"}},
        {"input": "Email must be valid format", "output": {"format": "email"}},
        {"input": "Date must be in ISO format", "output": {"format": "date"}},
        {"input": "DOB must be valid date", "output": {"format": "date"}},
        {"input": "Yes/No indicator: Y, N", "output": {"enum": ["Y", "N"]}},
    ]
}

# Use sample data for the selected domain
examples = SAMPLE_DATA[DOMAIN]
print(f"Loaded {len(examples)} examples for {DOMAIN} domain")

## 4. Prepare Dataset for T5

T5 expects text-to-text format. We'll format inputs as:
- Input: `"convert to json schema: {business_rule}"`
- Output: `"{json_constraint}"`

In [ ]:
def prepare_examples(examples):
    """Convert examples to T5 text-to-text format."""
    prepared = []
    for ex in examples:
        prepared.append({
            "input_text": f"convert to json schema: {ex['input']}",
            "target_text": json.dumps(ex['output'], separators=(',', ':'))
        })
    return prepared

# Prepare and create dataset
prepared_examples = prepare_examples(examples)
dataset = Dataset.from_list(prepared_examples)

# Split into train/validation (90/10)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(f"Train examples: {len(train_dataset)}")
print(f"Eval examples: {len(eval_dataset)}")
print(f"\nSample input: {train_dataset[0]['input_text']}")
print(f"Sample output: {train_dataset[0]['target_text']}")

## 5. Load Model and Tokenizer

In [ ]:
# Load tokenizer and model
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

print(f"Model: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")

## 6. Configure LoRA

LoRA adds small trainable matrices to the model while keeping the original weights frozen.

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q", "v"],  # Apply LoRA to attention layers
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

## 7. Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    """Tokenize inputs and targets."""
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        examples["target_text"],
        max_length=MAX_OUTPUT_LENGTH,
        truncation=True,
        padding="max_length",
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

print("Tokenization complete!")

## 8. Training

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none",  # Disable wandb
)

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("Starting training...")

In [ ]:
# Train the model
trainer.train()

## 9. Save the LoRA Adapter

In [ ]:
# Save the LoRA adapter
adapter_path = f"{OUTPUT_DIR}/final_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"Adapter saved to: {adapter_path}")

# Show adapter size
import os
total_size = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in os.listdir(adapter_path)
    if os.path.isfile(os.path.join(adapter_path, f))
)
print(f"Adapter size: {total_size / 1e6:.2f} MB")

## 10. Test the Model

In [ ]:
def predict(text, model, tokenizer):
    """Generate JSON Schema constraint from business rule."""
    input_text = f"convert to json schema: {text}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=MAX_INPUT_LENGTH, truncation=True)

    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
        model.cuda()

    outputs = model.generate(
        **inputs,
        max_length=MAX_OUTPUT_LENGTH,
        num_beams=4,
        early_stopping=True,
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    try:
        return json.loads(result)
    except json.JSONDecodeError:
        return {"raw": result, "error": "Could not parse JSON"}


# Test examples
test_cases = [
    "Age must be 18 or older",
    "Maximum 255 characters allowed",
    "Value must be between 0 and 100",
    "Status must be ACTIVE, INACTIVE, or PENDING",
    "Account number must be 10 digits",
    "Email must be valid format",
]

print("=" * 60)
print("MODEL PREDICTIONS")
print("=" * 60)

for test in test_cases:
    result = predict(test, model, tokenizer)
    print(f"\nInput: {test}")
    print(f"Output: {json.dumps(result)}")

## 11. Download the Adapter

Download the trained adapter to use in your application.

In [ ]:
# Zip and download the adapter
!zip -r adapter_{DOMAIN}.zip {OUTPUT_DIR}/final_adapter

from google.colab import files
files.download(f"adapter_{DOMAIN}.zip")

## 12. Loading the Adapter in Production

Here's how to load and use the adapter in your application:

In [ ]:
# Example: Loading adapter in production
PRODUCTION_CODE = '''
from transformers import T5Tokenizer, T5ForConditionalGeneration
from peft import PeftModel
import json

class T5ConstraintInterpreter:
    """T5-based interpreter using LoRA adapters."""

    def __init__(self, base_model="t5-small", adapter_path="./adapter_financial/final_adapter"):
        self.tokenizer = T5Tokenizer.from_pretrained(base_model)
        base = T5ForConditionalGeneration.from_pretrained(base_model)
        self.model = PeftModel.from_pretrained(base, adapter_path)
        self.model.eval()

    def interpret(self, business_rule: str) -> dict:
        """Convert business rule to JSON Schema constraint."""
        input_text = f"convert to json schema: {business_rule}"
        inputs = self.tokenizer(input_text, return_tensors="pt", max_length=128, truncation=True)

        outputs = self.model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True,
        )

        result = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        try:
            return json.loads(result)
        except json.JSONDecodeError:
            return {}


# Usage:
# interpreter = T5ConstraintInterpreter(adapter_path="./adapters/financial")
# result = interpreter.interpret("Balance must not be negative")
# print(result)  # {"minimum": 0}
'''

print(PRODUCTION_CODE)

## Next Steps

1. **Train more domains**: Change `DOMAIN` and re-run to create adapters for healthcare, ecommerce, etc.

2. **Use full dataset**: Replace sample data with full training data (500+ examples per domain)

3. **Integrate into your app**: Use the production code above to replace LLM API calls

4. **Benchmark**: Compare latency and accuracy vs LLM API

---

**Estimated costs:**
- Training: Free (Colab T4 GPU)
- Inference: Free (can run on CPU)
- Latency: 10-50ms (vs 500ms-2s for LLM API)